# TP3 — Sistema Multi-Agente RAG sobre CVs

**Autor:** Luciano  
**Curso:** CEIA — FIUBA — Clase 6  
**Stack:** Pinecone (vector store, namespace por agente) · Groq/Llama 3.1 (LLM) · HuggingFace MiniLM multilingüe (embeddings) · LangGraph (orquestación) · LangChain (RAG chain)

## Objetivo

Sobre el TP2 (RAG simple), implementar un **sistema de agentes** donde cada candidato tiene su propio agente especializado, con su propio namespace en Pinecone y su propio prompt. Un router decide a qué agente(s) mandar cada pregunta:

1. Si la pregunta menciona a una persona → responde solo su agente.
2. Si no menciona a nadie → responde el agente del alumno (default).
3. Si menciona a varias personas → responden todos los involucrados en paralelo y un nodo de síntesis combina las respuestas.

## ¿Por qué multi-agente y no un retriever único?

Con un único índice mezclando todos los CVs, el retriever puede traer 4 chunks del CV más "verboso" sobre un tema y ninguno de los otros, aunque los otros también respondan. Separando por agente, cada CV aporta su contexto sin competir por los top-k. Para preguntas comparativas esto es crítico.

## 0. Setup

Asumimos que ya está corrida la ingesta:

```bash
python -m src.ingestion --agent all --force
```

y que `.env` tiene `GROQ_API_KEY` y `PINECONE_API_KEY`.

In [40]:
import sys
from pathlib import Path
import time
import pandas as pd

# Permitir importar el paquete src/ desde notebooks/
sys.path.insert(0, str(Path.cwd().parent))

from src import config
from src.agents.orchestrator import preload_agents, run
from src.agents.registry import get_all_slugs, get_display_name, get_aliases
from src.evaluation import evaluate_query

print(f"Index Pinecone:    {config.PINECONE_INDEX_NAME}")
print(f"Embeddings:        {config.EMBED_MODEL}")
print(f"LLM:               {config.GROQ_MODEL}")
print(f"Top-K por agente:  {config.TOP_K}")
print(f"Agente default:    {config.DEFAULT_AGENT} ({get_display_name(config.DEFAULT_AGENT)})")
print()
print("Agentes registrados:")
for slug in get_all_slugs():
    print(f"  • {slug:5s} → {get_display_name(slug)}  (namespace cv_{slug})")

Index Pinecone:    cv-rag
Embeddings:        sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
LLM:               llama-3.1-8b-instant
Top-K por agente:  4
Agente default:    cv1 (Luciano Ceballos)

Agentes registrados:
  • cv1   → Luciano Ceballos  (namespace cv_cv1)
  • cv2   → Matías Ignacio Rossi  (namespace cv_cv2)
  • cv3   → Valeria Sofía Domínguez  (namespace cv_cv3)
  • cv4   → Daniel Alejandro Méndez  (namespace cv_cv4)


## 1. Arquitectura

```
                          INGESTA (una vez)
  ┌──────────┐     ┌──────────┐     ┌───────────┐     ┌──────────────┐
  │ data/cvs │ →   │ PyPDF +  │ →   │ Recursive │ →   │  MiniLM      │
  │  *.pdf   │     │ limpieza │     │ Splitter  │     │  multilg     │
  │ (1/agen) │     │          │     │ (500/50)  │     │  (384 dims)  │
  └──────────┘     └──────────┘     └───────────┘     └──────┬───────┘
                                                             │
                                                  ┌──────────▼─────────┐
                                                  │  Pinecone index    │
                                                  │  cv-rag            │
                                                  │  ├ ns cv_cv1       │
                                                  │  ├ ns cv_cv2       │
                                                  │  ├ ns cv_cv3       │
                                                  │  └ ns cv_cv4       │
                                                  └──────────┬─────────┘
                                                             │
                          RUNTIME (por query)                │
  ┌──────────┐                                               │
  │  query   │ ───┐                                          │
  └──────────┘    │                                          │
                  ▼                                          │
         ┌────────────────┐                                  │
         │  route_node    │ ← re.search() sobre aliases      │
         └────────┬───────┘                                  │
                  │                                          │
      ┌───────────┴─────────────┐                            │
      │ conditional edge        │                            │
      │ re.match(r'^single$')   │                            │
      └───┬─────────────────┬───┘                            │
          │ single          │ multi                          │
          ▼                 ▼                                │
  ┌─────────────┐    ┌──────────────────┐                    │
  │ rag_single  │    │ rag_multi        │ ←─ paraleliza con  │
  │ (1 agente)  │    │ (N agentes)      │    ThreadPool      │
  └──────┬──────┘    └────────┬─────────┘                    │
         │                    │ scopeo por namespace ────────┘
         │                    ▼
         │           ┌─────────────────┐
         │           │ synthesize_node │ ← LLM sobre respuestas
         │           └────────┬────────┘
         │                    │
         └──────────┬─────────┘
                    ▼
              respuesta final
```

**Decisiones de diseño clave:**
- **1 namespace por agente** (no un índice por agente) — es más eficiente en Pinecone y mantiene el aislamiento lógico. Cada `PersonAgent` está físicamente impedido de leer chunks de otro CV.
- **Routing por regex** (no por LLM) — determinístico, gratis, suficiente para nombres. Si hicieran falta resoluciones ambiguas se podría agregar una capa LLM.
- **Síntesis sobre respuestas, no sobre chunks** — el `synthesize_node` recibe lo que cada agente ya extrajo, no los chunks crudos. Esto evita que el LLM mezcle contextos y atribuya datos al candidato equivocado.

## 2. Pre-carga de agentes

Construye los retrievers y se conecta a Pinecone una sola vez para evitar latencia en la primera query.

In [41]:
preload_agents()
print(f"\n✓ {len(get_all_slugs())} agentes pre-cargados.")


✓ 4 agentes pre-cargados.


## 3. Inspección de los aliases del router

El router usa `re.search()` sobre estas listas de patrones para detectar a qué agente dirigir la pregunta. Si la query matchea con algún alias de un agente, ese agente se suma a la lista de agentes a invocar.

In [42]:
for slug in get_all_slugs():
    print(f"{get_display_name(slug)}  ({slug})")
    for alias in get_aliases(slug):
        print(f"   {alias}")
    print()

Luciano Ceballos  (cv1)
   \bluciano\b
   \bceballos\b
   \bluciano\s+ceballos\b

Matías Ignacio Rossi  (cv2)
   \bmat[ií]as\b
   \brossi\b
   \bmat[ií]as\s+(?:ignacio\s+)?rossi\b

Valeria Sofía Domínguez  (cv3)
   \bvaleria\b
   \bdom[ií]nguez\b
   \bvaleria\s+(?:sof[ií]a\s+)?dom[ií]nguez\b

Daniel Alejandro Méndez  (cv4)
   \bdaniel\b
   \bm[eé]ndez\b
   \bdaniel\s+(?:alejandro\s+)?m[eé]ndez\b



## 4. Caso 1 — Query con un nombre (rama `single`)

Cuando la pregunta menciona a una sola persona, el router selecciona su agente y `rag_single_node` lo invoca. Sin síntesis.

In [43]:
results = run("¿Qué experiencia laboral tiene Matias?")

for r in results:
    print(f"=== Agente: {r.display_name}  ({r.agent_slug}) ===")
    print(r.answer)
    print(f"\nFuentes recuperadas: {len(r.source_documents)} chunks")
    for d in r.source_documents:
        print(f"  • {d.metadata.get('source')} (pág. {d.metadata.get('page')})")

=== Agente: Matías Ignacio Rossi  (cv2) ===
Matías Ignacio Rossi: Tiene experiencia laboral como Sr. Network Engineer en Telecom Argentina desde enero de 2021 y actualidad. En este rol, ha liderado el despliegue de infraestructura de núcleo 5G para zonas metropolitanas y ha optimizado protocolos de enrutamiento BGP y OSPF en redes de transporte de fibra óptica. [cv2.pdf]

Fuentes recuperadas: 4 chunks
  • cv2.pdf (pág. 0.0)
  • cv2.pdf (pág. 0.0)
  • cv2.pdf (pág. 0.0)
  • cv2.pdf (pág. 0.0)


## 5. Caso 2 — Query sin nombre (default agent)

Si la query no matchea a nadie, `route_node` cae en el `DEFAULT_AGENT` (configurado a `cv1`, el agente del alumno). Es lo que pide la consigna del TP3:

> *"Por defecto, cuando no se nombra a nadie en la query, utilizar el Agente del alumno."*

In [44]:
results = run("¿Qué tecnologías domina?")

for r in results:
    print(f"=== Agente: {r.display_name}  ({r.agent_slug}) ===")
    print(r.answer)

=== Agente: Luciano Ceballos  (cv1) ===
Luciano Ceballos: Domina tecnologías como Python, SQL, PowerBi, AWS (para data engineering), Docker, Airflow, MLFlow, Databricks, numpy, pandas, matplotlib, seaborn, scipy, scikit-learn, nltk, opencv, pytorch. [cv1.pdf]


## 6. Caso 3 — Query con múltiples nombres (rama `multi` + síntesis)

Cuando la query menciona a 2+ personas:

1. `rag_multi_node` invoca a todos los agentes mencionados en paralelo (`ThreadPoolExecutor`).
2. Cada `PersonAgent` responde **solo sobre sí mismo** gracias a su prompt defensivo ("si la pregunta menciona a otras personas, ignoralas y respondé sobre tu candidato").
3. `synthesize_node` recibe las respuestas individuales y produce una respuesta comparativa con Groq.

El `RAGResult` final tiene `agent_slug="synthesis"` y un `display_name` del estilo *"Comparación — Luciano Ceballos vs Matías Ignacio Rossi"*.

In [45]:
results = run("Compará la formación académica de Valeria y Matías")

for r in results:
    print(f"=== {r.display_name}  ({r.agent_slug}) ===")
    print(r.answer)
    print(f"\nFuentes agregadas: {len(r.source_documents)} chunks")
    fuentes_por_cv = {}
    for d in r.source_documents:
        src = d.metadata.get('source', '?')
        fuentes_por_cv[src] = fuentes_por_cv.get(src, 0) + 1
    for src, n in fuentes_por_cv.items():
        print(f"  • {src}: {n} chunks")

=== Comparación — Matías Ignacio Rossi vs Valeria Sofía Domínguez  (synthesis) ===
La comparación de la formación académica de Valeria Sofía Domínguez y Matías Ignacio Rossi se puede realizar de la siguiente manera:

**Instituciones de estudio**

- Valeria Sofía Domínguez estudió en la Universidad Tecnológica Nacional.
- Matías Ignacio Rossi estudió en la Universidad Blas Pascal.

**Carreras académicas**

- Valeria Sofía Domínguez se graduó en Ingeniería en Sistemas de Información.
- Matías Ignacio Rossi se graduó en Ingeniería en Telecomunicaciones.

**Duración de los estudios**

- Valeria Sofía Domínguez cursó sus estudios desde 2013 hasta 2018, lo que equivale a 5 años.
- Matías Ignacio Rossi cursó sus estudios desde 2012 hasta 2018, lo que equivale a 6 años.

**Posgrados**

- Valeria Sofía Domínguez no menciona haber realizado un posgrado en su CV.
- Matías Ignacio Rossi realizó un posgrado en Ciberseguridad en el Instituto Tecnológico de Buenos Aires - ITBA en 2022.

En resumen, a

## 7. Conversación con follow-ups

El grafo acepta `chat_history` con mensajes LangChain (`HumanMessage`/`AIMessage`). El `PersonAgent` los usa en el prompt para dar respuestas coherentes a preguntas con referencias ambiguas ("¿y dónde trabajó?").

In [46]:
from langchain_core.messages import AIMessage, HumanMessage

# Turno 1
q1 = "¿Qué estudió Valeria?"
r1 = run(q1)
print(f"Q1: {q1}")
print(f"A1: {r1[0].answer}\n")

history = [
    HumanMessage(content=q1),
    AIMessage(content=r1[0].answer),
]

# Turno 2 — pregunta ambigua, depende del contexto del turno 1
q2 = "¿Y dónde trabaja actualmente?"
r2 = run(q2, chat_history=history)
print(f"Q2: {q2}")
print(f"A2: {r2[0].answer}")

Q1: ¿Qué estudió Valeria?
A1: Valeria Sofía Domínguez: Estudió Ingeniería en Sistemas de Información en la Universidad Tecnológica Nacional (2013 - 2018) [cv3.pdf].

Q2: ¿Y dónde trabaja actualmente?
A2: Luciano Ceballos: Actualmente trabaja como Especialista en Datos (Data Science-BI-ML) en Grido, desde noviembre de 2023 y en la actualidad [cv1.pdf].


## 9. Evaluación cuantitativa

El sistema tiene dos posibles fuentes de error: **routing** (¿el router eligió bien al agente?) y **retrieval** (¿el agente trajo del CV correcto?). `src/evaluation.py` mide ambos.

**Métricas:**
- `routing_accuracy` — fracción de queries donde el set de agentes seleccionados coincide exactamente con el ground truth.
- `precision@k` / `recall@k` — sobre los slugs de los chunks recuperados.
- `MRR` — posición del primer agente relevante en el ranking de chunks.

El eval set abajo está pensado para estresar al router en los tres casos de la consigna: single, default y multi.

In [47]:
from src.evaluation import EvalQuery, evaluate_all, summary_metrics

eval_set = [
    # === Single agent ===
    EvalQuery(
        question="¿Qué experiencia tiene Luciano con machine learning?",
        relevant_slugs=["cv1"],
    ),
    EvalQuery(
        question="¿Dónde estudió Matías?",
        relevant_slugs=["cv2"],
    ),
    EvalQuery(
        question="¿Qué tecnologías domina Valeria?",
        relevant_slugs=["cv3"],
    ),
    EvalQuery(
        question="¿En qué empresas trabajó Daniel?",
        relevant_slugs=["cv4"],
    ),
    # === Aliases por apellido ===
    EvalQuery(
        question="¿Qué hace Rossi profesionalmente?",
        relevant_slugs=["cv2"],
    ),
    EvalQuery(
        question="Hablame del perfil de Domínguez",
        relevant_slugs=["cv3"],
    ),
    # === Default agent (sin nombre → cv1) ===
    EvalQuery(
        question="¿Qué tecnologías sabe?",
        relevant_slugs=["cv1"],
    ),
    EvalQuery(
        question="Contame sobre su experiencia laboral",
        relevant_slugs=["cv1"],
    ),
    # === Multi-agent ===
    EvalQuery(
        question="Compará la formación de Daniel y Matías",
        relevant_slugs=["cv4", "cv2"],
    ),
    EvalQuery(
        question="¿Qué tecnologías comparten Valeria y Daniel?",
        relevant_slugs=["cv3", "cv4"],
    ),
    EvalQuery(
        question="Quién tiene más experiencia, Valeria, Matías o Daniel?",
        relevant_slugs=["cv3", "cv2", "cv4"],
    ),

]

print(f"Eval set: {len(eval_set)} queries")

Eval set: 11 queries


In [48]:
K = 4
rows = []

# Primer batch: queries single + default (1-8) — ~8 llamadas a Groq
print("=== Batch 1: queries single + default ===")
for i, eq in enumerate(eval_set[:8], 1):
    print(f"[{i}/11] {eq.question[:60]}...")
    rows.append(evaluate_query(eq, k=K))

print("\n⏸  Pausa de 60s para resetear el rate limit de Groq...")
time.sleep(60)

# Segundo batch: queries multi-agente (9-11) — ~15 llamadas a Groq
print("\n=== Batch 2: queries multi-agente ===")
for i, eq in enumerate(eval_set[8:], 9):
    print(f"[{i}/11] {eq.question[:60]}...")
    rows.append(evaluate_query(eq, k=K))

df_eval = pd.DataFrame(rows)
df_eval[["question", "selected_slugs", "relevant_slugs", "routing_correct", f"precision@{K}", f"recall@{K}", "mrr"]]

=== Batch 1: queries single + default ===
[1/11] ¿Qué experiencia tiene Luciano con machine learning?...
[2/11] ¿Dónde estudió Matías?...
[3/11] ¿Qué tecnologías domina Valeria?...
[4/11] ¿En qué empresas trabajó Daniel?...
[5/11] ¿Qué hace Rossi profesionalmente?...
[6/11] Hablame del perfil de Domínguez...
[7/11] ¿Qué tecnologías sabe?...
[8/11] Contame sobre su experiencia laboral...

⏸  Pausa de 60s para resetear el rate limit de Groq...

=== Batch 2: queries multi-agente ===
[9/11] Compará la formación de Daniel y Matías...
[10/11] ¿Qué tecnologías comparten Valeria y Daniel?...
[11/11] Quién tiene más experiencia, Valeria, Matías o Daniel?...


,question,selected_slugs,relevant_slugs,routing_correct,precision@4,recall@4,mrr
0,¿Qué experiencia tiene Luciano con machine lea...,[cv1],[cv1],1,1.0,1.0,1.0
1,¿Dónde estudió Matías?,[cv2],[cv2],1,1.0,1.0,1.0
2,¿Qué tecnologías domina Valeria?,[cv3],[cv3],1,1.0,1.0,1.0
3,¿En qué empresas trabajó Daniel?,[cv4],[cv4],1,1.0,1.0,1.0
4,¿Qué hace Rossi profesionalmente?,[cv2],[cv2],1,1.0,1.0,1.0
5,Hablame del perfil de Domínguez,[cv3],[cv3],1,1.0,1.0,1.0
6,¿Qué tecnologías sabe?,[cv1],[cv1],1,1.0,1.0,1.0
7,Contame sobre su experiencia laboral,[cv1],[cv1],1,1.0,1.0,1.0
8,Compará la formación de Daniel y Matías,"[cv2, cv4]","[cv4, cv2]",1,1.0,1.0,1.0
9,¿Qué tecnologías comparten Valeria y Daniel?,"[cv3, cv4]","[cv3, cv4]",1,1.0,1.0,1.0


In [49]:
summary = summary_metrics(df_eval, k=K)
for metric, value in summary.items():
    if isinstance(value, float):
        print(f"{metric:25s}: {value:.3f}")
    else:
        print(f"{metric:25s}: {value}")

routing_accuracy         : 1.000
precision@4_mean         : 1.000
recall@4_mean            : 1.000
mrr_mean                 : 1.000
n_queries                : 11


### Análisis de errores

Si alguna métrica no da 1.0, conviene mirar las queries que fallaron para diagnosticar si el problema está en el router (aliases insuficientes) o en el retrieval (chunks irrelevantes).

In [50]:
errores_routing = df_eval[df_eval["routing_correct"] == 0]
if len(errores_routing) == 0:
    print("✓ Routing perfecto en todas las queries.")
else:
    print(f"✗ {len(errores_routing)} queries con routing incorrecto:")
    for _, row in errores_routing.iterrows():
        print(f"\n  Q: {row['question']}")
        print(f"     esperado:    {row['relevant_slugs']}")
        print(f"     seleccionado: {row['selected_slugs']}")

✓ Routing perfecto en todas las queries.


## 10. Conclusiones

1. **El sistema cumple las tres funcionalidades pedidas:** un agente por persona, default agent al no mencionar a nadie, y trato adecuado de queries multi-CV (contexto separado por persona + síntesis).
2. **El routing por regex es robusto** para nombres propios y aliases definidos en `config.AGENTS`. Para queries muy ambiguas (ej. *"el más experimentado"*) el sistema cae al default agent, que es el comportamiento conservador esperado.
3. **La síntesis sobre respuestas individuales** (no sobre chunks) reduce el riesgo de alucinaciones cruzadas entre candidatos. Cada agente actúa como una fuente confiable sobre *su* persona, y el nodo de síntesis solo reorganiza esa información.
4. **Las métricas de IR a nivel agente** (precision/recall/MRR sobre slugs) permiten diagnosticar si los errores vienen del router o del retrieval, lo cual orienta dónde iterar.